# openpilot supercombo on KITScenes: the scene sweep's config, run by paddock on every keyframe

> **Outputs are not committed: run the notebook to see them.** Every figure, table and number the text refers to comes from a run, and git keeps none of them: `tests/test_notebooks_are_bare.py` holds this file to that.
>
> 1. **Once:** `make setup`, then `uv pip install --python .venv/bin/python -e ".[analysis]"` for Jupyter and what the notebook draws with.
> 2. **The site:** `paddock.toml` (or `PADDOCK_CONFIG`) says where `{runs}` and the data are. It reads the records of the three runs it compares (§1) under `{runs}`, and `gate()` answers from this site's `paddock.toml` pairing tables.
> 3. **Run it:** run all on the repo's `.venv` kernel, or headless from the repo root: `.venv/bin/jupyter execute --output='{notebook_name}_all_executed' tools/models/openpilot/analysis/openpilot_drv_supcombo_kitscenes_config_run.ipynb`. That writes an executed copy beside this file, which git ignores. Never `--inplace`: it writes the outputs into this file, and `make check` fails on that. A run in Jupyter or VS Code saves them here too; strip it before committing, with `.venv/bin/python -m tools.strip_notebooks tools/models/openpilot/analysis/openpilot_drv_supcombo_kitscenes_config_run.ipynb`.
>
> **What it needs.** No model, no node, no GPU: seconds. Run the open-loop notebook first: with `TAGS = "auto"` this one takes its tags from that notebook's export, and without the export it says so and goes on without them.

The [scene-sweep notebook](openpilot_drv_supcombo_kitscenes_scene_sweep.ipynb)'s `CONFIG` was chosen through the analysis node on a few dozen keyframes. It does four things:

- it feeds the model the base front camera instead of the ring's, cut to 55° at comma's 1164x874;
- it shows the road as from a camera 1.22 times lower;
- it tilts the view 1.7° down;
- it sends the route as a turn desire.

Paddock then ran it itself, from a pairing table in the site config, on the 223 keyframes the stock run scored. This notebook reads that run beside two recorded ones:

- **stock** (`20260930T154018-242869`): nothing set. It is the [open-loop notebook](openpilot_drv_supcombo_kitscenes_openloop_analysis.ipynb)'s baseline.
- **intent** (`20260930T014333-d329a6`): the route as a desire, on the ring camera. It is the [four-questions note](openpilot_drv_supcombo_kitscenes_four_questions.md)'s Q1.
- **config** (`20261008T004723-a369d1`): the scene sweep's `CONFIG`. It re-runs `20261005T205400-a369d1`, equals it on all 223 keyframes, and superseded it.

For these runs it gives:

1. **Parameters.**
2. **What ran.** Each run's settings, and a check that paddock's run is the node's on the two keyframes both scored. Then the gate, set before the run: the config would become this site's default only if it beat stock.
3. **Overall.** The change on every keyframe, on the ones the config was tuned on, and on the ones it never saw, each with a 95 % interval over scenes. Then every metric, and a table of every keyframe.
4. **Per scene.** Which scenes got better and which got worse.
5. **The bad keyframes.** The config's worst plans and its largest losses, with the three largest drawn.
6. **Where.** By the car's speed, where most of the change is, then by the tags.
7. **The image against the desire.** The change on turns, split into the desire's share and the image's.
8. **What the model read.** Its camera height and its speed, against the four-questions note.
9. **One keyframe** (`FOCUS`).
10. **What this shows.**

Nothing here runs the model. Every number is read from the runs' records, so the notebook re-executes in seconds and cannot drift from what paddock recorded. The code is in [the config-run lib](openpilot_drv_supcombo_kitscenes_config_run_lib.py). The [note](openpilot_drv_supcombo_kitscenes_config_run.md) states the findings, with the strength of each claim.

**The config was chosen on the set it is scored on**, by inspecting one scene of it, dbba76d9, and the patterns the earlier analysis found; none of it was fitted. That analysis looked at 90 of the 223 keyframes, the "tuned" set, and part 3 lists where. The other 133 are held out. Every one of them is straight, because the desire was scored on every turn.

In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

from paddock.config import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tools/models/openpilot/analysis"))  # the libs, beside this notebook
import openpilot_drv_supcombo_kitscenes_config_run_lib as C

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)  # the knobs a run set, and stock's failure modes, in full

## 1. Parameters

- **`RUNS`**: `{arm: run id}`, each a run of openpilot under `results/kitscenes/openpilot/`.
    - `"stock"` and `"config"` are required. `"intent"` is needed only by part 7.
    - Point `"config"` at another run to read it the same way. Every run must have scored the same keyframes.
- **`METRIC`**: what parts 3 to 7 compare. It is one of these, each in metres over the benchmark's 3 s:
    - `"ade"`;
    - `"fde_1s"`, `"fde_2s"` or `"fde_3s"`;
    - `"lat"` (lateral MAE) or `"lon"` (longitudinal MAE).
- **`ON`**: the keyframes parts 3 to 6 and 8 read. It can be:
    - a subset part 3 names, such as `"all"`, `"held out"`, `"turns"` or `"sweep60"`;
    - a keyframe, such as `"93bcdd59/40"`;
    - a scene prefix, such as `"3a616875"`;
    - a list of any of these.
- **`FOCUS`**: the keyframe part 9 draws.
- **`SORT`**: the column that sorts part 3's table of every keyframe. `"change"` is config − stock in `METRIC`.
- **`OVERRIDES`**: any of `C.PARAMS`, such as `{"tie_m": 0.1}`, `{"far_turn_m": 30.0}` or `{"speed_edges": (0, 6, 10, float("inf"))}`. Part 2 prints them all.
- **`TAGS`**: the stock run's tags table, which the open-loop notebook's §18 export writes.
    - `"auto"` finds it under `{runs}`, and a path names it.
    - `None` goes without it. Part 6 then has only this notebook's own columns.
- **`WIDGETS`**: dropdowns in part 6. They are off by default because VS Code cannot draw a saved widget. Set `WIDGETS` here, or `PADDOCK_ANALYSIS_WIDGETS=1`, to turn them on.

In [ ]:
RUNS = dict(C.RUNS)  # stock, intent and config, as above
METRIC = "ade"
ON = "all"
FOCUS = "93bcdd59/40"  # the largest gain: a straight at 13.7 m/s that stock planned from 9.7
SORT = "change"
OVERRIDES = {}
TAGS = "auto"
WIDGETS = os.environ.get("PADDOCK_ANALYSIS_WIDGETS") == "1"

## 2. What ran

Each run's settings and what openpilot's adapter reported about itself (`run.json`'s `provenance.policy`), one column per arm.

The config differs from stock in exactly what the pairing table sets: the front camera, the command source and four of the adapter's knobs. ONNX is pinned because the tinygrad backend refuses the cut. The weights, frames and sample list are the same.

In [ ]:
C.WIDGETS = WIDGETS
study = C.Study(RUNS, tags=TAGS, overrides=OVERRIDES)
print("params, after OVERRIDES:", study.p)
study.provenance()

**The oracle.** The scene-sweep notebook scored dbba76d9/40 and /130 through the analysis node, under stock and under its `CONFIG`. Paddock's runs must give the same ADE to the last bit. They use the same weights, frames, warp and desire, reached through paddock's adapter rather than the node's.

A `False` would mean paddock did not run the config the notebook chose. The check applies only to the runs in `C.RUNS`.

In [ ]:
oracle = study.oracle()
if RUNS == C.RUNS:  # the oracle is the scene sweep's CONFIG; another config run has no reason to match it
    assert oracle.equal.all(), "paddock's run is not the node's on the oracle's keyframes"
oracle

**The gate.** It was set before the run: the config becomes this site's default only if its mean ADE over the 223 keyframes is below stock's. That is the number `paddock report` ranks by.

`paddock run here measures as` asks a separate question.
- It builds the config that a plain `paddock run --bench kitscenes --model openpilot` would record under this site's `paddock.toml`, the way the runner builds it.
- It then names the run that config equals. That is `config` while the pairing tables are there, and `stock` once they are deleted.

The config run here is the re-run, which is valid and ranked: the README's OpenPilot KITScenes figure. Its route hint is recorded in its `config`, as the benchmark's other route-conditioned rows have theirs.

In [ ]:
study.gate()

## 3. Overall

Where the config's knobs were chosen or scored (`C.CHOSEN`, `C.SCORED`). The lib restates each set from the cell of the notebook that defines it.

- **`sweep60`**: the open-loop notebook's height sweep (its §15) ran the first 60 keyframes by scene. s\* = 1.22 is what it measured there.
- **`ten`**: the image-calibration notebook's ten turns. There the height and the pitch were set on grids.
- **`five`**: five of the ten, in the image-tweak notebook. There the pitch, the camera, the cut and the desire were set by hand.
- **`scene sweep`**: dbba76d9. There the `CONFIG` was checked on every frame from 40 to 159.
- **`every turn`**: the desire was scored on all 39 turns before it was fixed, by the intent run and by the calibration notebook's §13.

The subsets built from these:
- **`tuned`** is the union of these sets.
- **`held out`** is everything else: 133 keyframes, every one straight.
- **`held-out scenes`** also drops every keyframe of a scene that had a tuned one, in case a scene's look carries from one keyframe to the next.
- **`turns not chosen on`** are the turns outside `CHOSEN`. Their desire was scored, but no knob was chosen on them.

In [ ]:
study.subset_table()

The config against stock on each subset, in metres.

- `change` is config − stock, so negative is better.
- It has a 95 % interval from resampling scenes, because the keyframes of one scene are not independent.
- `better` and `worse` count the keyframes that moved by more than `tie_m`.
- The model is deterministic here: the oracle matches across two separate pipelines to the last bit. So the interval is the whole noise reference. It says how much the change depends on which scenes happen to be in the set.

In [ ]:
study.overall(metric=METRIC).round(3)

On each set a knob was chosen or scored on, one at a time:

In [ ]:
study.overall(list(C.SEEN), metric=METRIC).round(3)

Every metric on `ON`:

In [ ]:
every = pd.DataFrame({C.METRICS[m]: study.paired(ON, metric=m) for m in C.METRICS}).T
every.astype({c: int for c in ("keyframes", "scenes", "better", "worse", "tie")}).round(3)

**Every keyframe**, one row each, sorted by `SORT`. Each row says what the keyframe is, then gives each arm's metrics and readings as `<name>_<arm>`.

- `v` is the pose speed.
- `v0_*` is the first 0.5 s of the GT or of a plan.
- `cam_h_*` is the camera height the road head read.
- `dpsi_*` is the heading change toward the route's side, over 3 s (over 10 s for `dpsi_10s_*`).

Filter it like any DataFrame, e.g. `scores[scores.route == "left"]`.

In [ ]:
scores = study.scores.assign(change=study.scores[f"{METRIC}_config"] - study.scores[f"{METRIC}_stock"])
scores = scores[study.mask(ON)].sort_values(SORT)
with pd.option_context("display.max_rows", len(scores)):
    display(scores.round(3))

## 4. Per scene

- **Left:** every scene's change in mean `METRIC`, ranked. Dark bars are scenes with a tuned keyframe.
- **Right:** each scene's mean under the config against stock's, on log axes. Below the diagonal, the config is better.

In [ ]:
study.plot_scenes(metric=METRIC, on=ON)

In [ ]:
scenes = study.by_scene(metric=METRIC, on=ON)
print("the 12 most improved scenes, then the 12 most worsened")
display(scenes.head(12).round(3))
scenes.tail(12)[::-1].round(3)

## 5. The bad keyframes

The config's worst plans, then its largest losses against stock.

`stock's failure modes` are the open-loop notebook's tags on the stock run. They say what went wrong with stock's plan there, not with the config's.

In [ ]:
study.worst(12, rank="config", metric=METRIC, on=ON).round(3)

In [ ]:
study.worst(15, rank="change", metric=METRIC, on=ON).round(3)

The three largest losses, drawn. Each has three panels:
- every arm's plan against the GT, from above: solid within the benchmark's 3 s, dotted beyond;
- the keyframe from the ring's front camera, which stock and intent saw;
- the keyframe from the base camera, with the config's cut boxed.

In [ ]:
for key in study.worst(3, rank="change", metric=METRIC, on=ON).index:
    study.show(key)

## 6. Where: the car's speed, then the tags

Each band of the pose speed (`speed_edges`):
- **Left:** each arm's mean `METRIC`.
- **Right:** each arm's median plan for the first 0.5 s, against the GT's. The dashed line at 1 is a plan that starts at the right speed.

In [ ]:
study.plot_speed(metric=METRIC, on=ON)

In [ ]:
study.by_speed(metric=METRIC, on=ON).round(3)

The stock run's tags break the change down by:
- the route;
- the GT's lateral and longitudinal shape;
- the tags' speed band;
- the lane markings;
- the setting;
- whether the car is already turning.

Any column of `study.scores` or `study.tags` works, e.g. `study.breakdown("split")`.

In [ ]:
BY = ["route", "gt_lat", "gt_long", "speed_band", "lanes", "ctx_location", "ego_turning"]
for col in BY:
    if col in study.scores or col in study.tags:
        display(study.breakdown(col, metric=METRIC, on=ON).round(3))
    else:
        print(f"{col}: not here; it is a tags column, and TAGS found no table")

In [ ]:
study.explore(metric=METRIC)

## 7. The image against the desire

- **On a straight route** the intent run sends no desire, so it is stock to the last bit. There, config − stock is the image alone.
- **On a turn**, intent − stock is the desire on the ring camera, and config − intent is the image with the desire sent. The two steps add up to `both`.

Near and far turns are split at `far_turn_m`: the distance from which the note's Q1 found the desire costing ADE.

In [ ]:
study.decompose(metric=METRIC).round(3)

## 8. What the model read

The four-questions note found three things about stock:
- it reads its camera height in one of two modes, near 1.39 m or near 1.62 m (Q2);
- its pose head reads the speed 12 % slow (Q4);
- a height scale s\* = 1.22, measured on the ring camera, makes the head read the speed right (Q4).

The config shows the road as from a camera 1.22 times lower, but on the base camera. First, how high each front camera is, and how high the config's scale shows the road from:

In [ ]:
study.heights().round(3)

Then what each arm read, as medians: on `ON`, and on the turns alone.

In [ ]:
study.readings(ON)

In [ ]:
study.readings("turns")

In [ ]:
study.plot_readings(ON)

## 9. One keyframe (`FOCUS`)

Its plans and frames, then its row of part 3's table.

In [ ]:
study.show(FOCUS)
study.scores[study.mask(FOCUS)].T

## 10. What this shows

Read off the cells above on 2026-10-05. Rerunning with other `RUNS`, `ON` or `OVERRIDES` changes the numbers but not this text.

**Measured** (changes are config − stock, with 95% CIs from resampling scenes):

- **The run reproduces the scene sweep, and it beats stock.** On dbba76d9/40 and /130, paddock's ADE equals the analysis node's to the last bit, for stock and config alike (part 2). Over all 223 keyframes, ADE fell from 1.595 to 1.248 m. That is under stock's 1.595359, the bar fixed before the run, so this site's `paddock.toml` now carries the two pairing tables and a plain run measures this configuration. That run is ranked, with its route hint recorded; its re-run, identical on every keyframe, is OpenPilot's KITScenes figure in the README (part 2).
- **Most of the gain is on keyframes no knob was chosen on** (part 3).
    - Overall: −0.347 m (CI −0.551 to −0.136; −21.8%), better on 129 keyframes and worse on 83.
    - On the 90 tuned keyframes, −0.063 (CI −0.404 to +0.259), which cannot be told from no change.
    - On the 133 held out, −0.540 (CI −0.810 to −0.287; −33%). On the 110 of those in scenes with no tuned keyframe, −0.568.
    - The sets the knobs were picked on did not all improve: sweep60 went −0.213, ten +0.280 and five +0.260.
- **Slow keyframes got worse and fast ones better, crossing at 6 m/s** (part 6).
    - The config raises the plan's start speed at every speed. Above 2 m/s, stock started at 0.895–0.930 of the GT's start speed. The config starts at 1.03 of it above 8 m/s, 1.06 at 6–8 m/s, and 1.14–1.15 at 2–6 m/s.
    - Above 6 m/s, ADE falls in every band, most at 12+ m/s: 2.934 → 1.485 m (−1.449, CI −2.312 to −0.571).
    - Below 6 m/s it rises: +0.057 at 0–2 m/s, +0.222 at 2–4 and +0.324 at 4–6. Only the last has a CI clear of zero (+0.016 to +0.625).
- **The gain is along the path, and across it the config is worse** (part 3).
    - Longitudinal MAE fell from 1.526 to 1.136 m (−25.5%).
    - Lateral MAE rose from 0.286 to 0.339 m (+0.053, CI +0.028 to +0.082; +18.6%). It got worse on 81 keyframes and better on 42.
    - FDE improved at every horizon. At 3 s it fell from 2.904 to 2.355 m.
- **Turns got worse, left turns most** (parts 3 and 6).
    - On the 39 turns: +0.234 m (CI −0.222 to +0.681), 19 better and 19 worse.
    - By the GT's shape, `turn_L` (14 keyframes) went +0.675 (CI +0.014 to +1.232; 5 better, 9 worse). It is the only shape whose CI lies above zero. `straight` went −0.939 (−49%) and `curve_L` −0.454.
    - Where the ego is already yawing (`ego_turning`, 56 keyframes), +0.140. Where it is steady (167), −0.511.
- **On turns, the image costs on near ones and the desire on far ones** (part 7). Intent equals stock to the last bit on all 184 straight keyframes, so the −0.470 on straights is the image alone. On turns, the two can be told apart:
    - **Within 40 m (30 turns).** The desire alone (intent − stock, on the ring camera) changes ADE by −0.066 (CI −0.299 to +0.156). The image, given the desire (config − intent), adds +0.289 (CI −0.163 to +0.694), worse on 20 and better on 8.
    - **Beyond 40 m (9 turns).** The desire alone costs +1.540 (CI +0.585 to +2.418; 7 worse). The image wins back −1.266 (CI −1.974 to −0.622; 8 better), leaving +0.274 against stock.
- **Where it helped and where it hurt** (part 4). Of 140 scenes, 86 got better, 45 worse, and 9 moved less than 0.05 m.
    - The 12 most improved scenes average 7.6–14.6 m/s. All are straights apart from one left turn in 873eebf4. The top three are 2381584f (5.058 → 1.405 m), 97a94a07 (3.827 → 0.693) and c71eb5a2 (3.063 → 0.462).
    - Seven of the 12 most worsened average under 5 m/s. The top three are b8002a79 (two right turns, 1.427 → 3.423), fdcc7efa (a left turn, 0.710 → 2.645) and ed483615 (a straight at 3.8 m/s, 0.857 → 2.683).
- **The especially bad keyframes** (part 5).
    - **873eebf4/130** is the config's worst plan, at 6.077 m: a left turn 9 m ahead at 2.9 m/s. Stock planned it worse (6.578), so the config did not cause it.
    - **9ffae20c/310** is the largest loss: +3.951 m (1.120 → 5.071), on a held-out straight at 8.9 m/s. From its figure, the GT slows and covers about 24 m in 4 s, while the config plans about 31 m in 3 s. Stock's plan stays near the GT.
    - **b8002a79/130** is a right turn 79 m ahead at 14.6 m/s: 0.716 → 3.775 m, with intent at 4.539. Told of the turn, the model turns its plan long before the road does, and the image wins back only part of that.
    - **3a616875**: all five of its keyframes are tuned, and three are among the 15 largest losses. At /40, a straight at 4.75 m/s, ADE went 0.644 → 2.980 m: the config plans about 17 m in 3 s where the GT covers 16 m in 4 s. /220 went +2.232 and /310 +1.707.
    - Stock's own plan was already too fast (its `too_fast_cruise` tag) on 5 of the 15 largest losses. There the config's faster start adds to an error stock already had.
- **The speed the model reads went from slow to slightly fast** (part 8).
    - Pose head / pose speed went 0.877 → 1.033.
    - The plan's first step / the GT's went 0.909 → 1.051, and its mean absolute error 0.845 → 0.558 m/s.
    - On the 39 turns the first step overshoots: 0.921 → 1.140, and its error rose from 0.743 to 0.877 m/s.
- **The camera-height reading is no longer bimodal** (part 8). The road head reads every keyframe in the low mode: 223 of 223 under 1.5 m, median 1.372 m, against stock's 130. The four-questions note measured that this mode hardly follows the image, and argued that it is a floor near 1.38 m. The base camera's map height over 1.22 is also 1.372 m, so this run cannot tell the floor from a true reading.
- **The base camera sits 0.20 m below the ring's** (part 8): map heights 1.674 against 1.875 m, calibration heights 1.816 against 2.017 m. At 1.22, the base camera's road is shown as from 1.372 m (map), against the ring's 1.537 m.

**Argued, not tested:**

- **1.22 was measured on the other camera.**
    - The four-questions note's rule is to show the road as from about 1.55 m. s\* = 1.22 was fitted on ring frames, where it does that.
    - On the base camera, 1.22 shows the road from 1.37 m, and a road seen from lower reads as faster. That fits the sign of the over-read: pose head 1.033, plan start 1.051.
    - It does not obviously explain the over-read's pattern: 1.14–1.15 at 2–6 m/s, but about 1.03 above 8 m/s.
    - By the same rule, the base camera wants s ≈ 1.08 (1.674 / 1.55). Testing that is one more run, and none was made.
- **The held-out number is a straight-road number.** Every turn was used to choose the desire, so all 133 held-out keyframes are straight. Turns have no held-out estimate. The nearest is the 23 turns the image knobs were not chosen on, though their desire was still scored: +0.196 (CI −0.321 to +0.806).
- **The turn numbers assume an oracle's route.** The config reads the route from the GT path (`command_source = "gt"`), as intent does, and a car would not have that. On far turns this hint arrives 40 m or more ahead, which is where the desire costs most.
- **About 7.4% of big_img is filler.** The 55° cut is narrower than big_img's 58.7°, so `big_img in bounds` is 0.926, and the rest is smeared from the cut's border. Whether the model reads that margin was not tested.